# Modelado — comparación de modelos (forecast t+1)

Comparación de **regresión lineal**, **Random Forest** y **XGBoost** para predecir
`prod_pet` del mes siguiente (`y_next`).

**Protocolo (ADR-028):**
- Se **entrena en `train`** y se **compara en `val`**; `test` queda intacto.
- Métricas: **RMSE** (m³, penaliza errores grandes) y **R²** (varianza explicada).
- Todo el preprocesamiento (one-hot, imputación, normalización) se **ajusta solo
  en train** y se aplica a val → sin leakage val→train. Las features ya son
  leak-safe en el tiempo (ver notebook 03).

La lógica reutilizable vive en `ml/modeling.py`; acá solo se orquesta.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml import modeling

pd.set_option("display.width", 120)

## 1. Dataset y matriz de features

`build_feature_matrix` arma el dataset (notebook 03) y aplica el one-hot de las
categóricas **ajustado en train**. Las features = todas las columnas menos las
claves (`idpozo`, `periodo`, `periodo_objetivo`, `split`) y el target `y_next`.

In [2]:
ds_enc, feature_cols = modeling.build_feature_matrix()

print(f"Filas: {len(ds_enc):,}  |  Features: {len(feature_cols)}")
print(ds_enc["split"].value_counts().reindex(["train", "val", "test"]).to_string())

Filas: 319,560  |  Features: 380
split
train    223621
val       47883
test      48056


## 2. Split train/val + preprocesamiento (anti-leakage)

- **Imputación** de NaN con la **mediana de train** (Random Forest y regresión
  lineal no toman NaN). Los NaN vienen de las features de historia (primeros
  meses de cada pozo).
- **Normalización** (media 0, desvío 1) **solo para la regresión lineal**
  (sensible a la escala); los árboles no la necesitan.

Ambos estadísticos (mediana, media/desvío) se ajustan **solo en train**.

In [3]:
X_train, y_train, X_val, y_val = modeling.split_train_val(ds_enc, feature_cols)
print("train:", X_train.shape, " | val:", X_val.shape)

train: (223621, 380)  | val: (47883, 380)


## 3. Entrenamiento y comparación en val

`train_eval_models` entrena los 3 modelos en train (imputado; + normalizado para
el lineal) y los evalúa en val. Agregamos la **persistencia** (ŷ(t+1) = prod_pet(t))
como referencia: es el baseline a batir (ADR-029).

In [4]:
tabla, modelos = modeling.train_eval_models(X_train, y_train, X_val, y_val)

pers = modeling.persistence_val(ds_enc)
comparacion = pd.concat(
    [tabla, pd.DataFrame([{"modelo": "persistencia (baseline)", **pers}])],
    ignore_index=True,
).sort_values("val_rmse").reset_index(drop=True)
comparacion.round(3)

,modelo,val_rmse,val_r2
0,xgboost,238.403,0.893
1,persistencia (baseline),250.589,0.881
2,random_forest,256.009,0.876
3,reg_lineal,609.018,0.300


**Lectura.**

- **XGBoost** es el mejor (RMSE más bajo, R² más alto) y **supera a la
  persistencia**; la **regresión lineal** también la supera por poco. El
  **Random Forest**, con estos hiperparámetros, queda **por debajo** de la
  persistencia.
- Los **R² son todos altos (~0.88–0.89)**: la mayor parte de la varianza de
  `prod_pet(t+1)` ya la explica `prod_pet(t)` (autoregresión fuerte), así que el
  margen de mejora sobre la persistencia es **chico** y se juega en los errores
  grandes (RMSE).
- Que el Random Forest quede atrás es típico de esta configuración (muchas
  dummies ralas + `max_features="sqrt"` + hojas grandes → promedia de más);
  mejora bastante con tuning.

**Próximos pasos:**
1. **Tuning** de XGBoost sobre val (profundidad, learning rate, n_estimators).
2. Probar **`log1p`** sobre el target: la fuerte asimetría infla el RMSE y suele
   mejorar con la transformación.
3. Loguear las corridas en **MLflow** (consistente con los baselines, ADR-030).

## 4. Tuning de hiperparámetros con CV temporal

¿Vale la pena un grid search? **Sí**, pero **no** con el `KFold` por defecto: baraja
las filas al azar, así que folds podrían entrenar con meses *posteriores* a los que
validan → **leakage futuro→pasado**. Por ser serie temporal usamos:

- **CV temporal (expanding window) por mes** (`modeling.time_series_folds`): en cada
  fold el bloque de validación es **posterior** a todo el train del fold.
- **`Pipeline` (imputer + scaler) dentro del search**: `GridSearchCV` reajusta el
  preprocesamiento **solo con el train de cada fold** → sin leakage de estadísticos.
- La búsqueda se hace **dentro de train**; **val sigue intacto** como comparación final.

Modelos: **Ridge** (tuneando `alpha` = lambda L2), **Random Forest** y **XGBoost**.

Cada modelo muestra una **barra de progreso** (`tqdm`) que avanza por cada
combinación de hiperparámetros e indica el mejor RMSE de CV hasta el momento.

> ⚠️ Esta celda entrena muchos modelos (sobre todo el Random Forest): puede tardar
> varios minutos. La barra te deja ver cómo evoluciona.

In [5]:
tabla_cv, searches = modeling.tune_all(ds_enc, feature_cols, n_splits=4)
tabla_cv

ridge (6 combos):   0%|          | 0/6 [00:00<?, ?it/s]

random_forest (8 combos):   0%|          | 0/8 [00:00<?, ?it/s]

xgboost (12 combos):   0%|          | 0/12 [00:00<?, ?it/s]

,modelo,cv_rmse,best_params
0,ridge,215.925833,{'model__alpha': 1000.0}
1,random_forest,221.364429,"{'model__max_depth': 16, 'model__max_features'..."
2,xgboost,230.480544,"{'model__colsample_bytree': 0.8, 'model__learn..."


Mejores hiperparámetros por modelo (los de menor RMSE en la CV temporal):

In [6]:
for nombre, search in searches.items():
    print(f"{nombre}: cv_rmse={-search.best_score_:.3f}")
    print("  ", search.best_params_)

ridge: cv_rmse=215.926
   {'model__alpha': 1000.0}
random_forest: cv_rmse=221.364
   {'model__max_depth': 16, 'model__max_features': 0.3, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}
xgboost: cv_rmse=230.481
   {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__n_estimators': 400, 'model__subsample': 0.8}


### 4.1 Evaluación de los modelos tuneados en val

Cada `best_estimator_` (reentrenado en **todo** train con sus mejores
hiperparámetros) se evalúa sobre **val**, que no se tocó durante el tuning. Comparamos
contra la persistencia y, en la sección 3, contra los modelos sin tunear.

In [7]:
comparacion_tuned = modeling.eval_searches_on_val(searches, ds_enc, feature_cols)
comparacion_tuned.round(3)

,modelo,val_rmse,val_r2
0,random_forest,226.875,0.903
1,xgboost,236.561,0.894
2,ridge,244.167,0.887
3,persistencia (baseline),250.589,0.881


### 4.2 Métricas en train y val con la mejor config

Toma la **mejor configuración** hallada por modelo (el `best_estimator_`, ya
reentrenado en todo train) y mide RMSE/R² en **train** y en **val**. Ver las dos
juntas muestra el **gap train→val**: si train es mucho mejor que val, el modelo
está sobreajustando.

In [8]:
metricas_train_val = modeling.eval_searches_train_val(searches, ds_enc, feature_cols)
metricas_train_val.round(3)

,modelo,train_rmse,train_r2,val_rmse,val_r2
0,random_forest,171.565,0.954,226.875,0.903
1,xgboost,180.696,0.949,236.561,0.894
2,ridge,282.820,0.875,244.167,0.887
3,persistencia (baseline),309.990,0.849,250.589,0.881


**Lectura.** _(completar al correr)_

- Comparar el `val_rmse`/`val_r2` de cada modelo tuneado contra: (a) la persistencia
  y (b) su versión sin tunear de la sección 3.
- Para Ridge, ver qué `alpha` ganó: `alpha` alto = más regularización (modelo más
  simple); cercano a 0 = casi una regresión lineal sin penalización.
- Si el RMSE de val queda muy por debajo del de CV, puede haber sobreajuste a val al
  elegir hiperparámetros; el juicio final se hace una sola vez sobre **test**.